# Population-weighted distance-decay distribution
First run [3_population_distribution.ipynb](3_population_distribution.ipynb) through its aggregate export section. For special demand, also run SD4’s university and airport exports. Run this notebook with the repository root as the working directory.

Workplace-area/residence-region OD targets constrain allocation. Within each pairing, workplace probabilities are proportional to remaining job population times `exp(-distance / DISTANCE_SCALE)`, with origins sampled by remaining residents. Capacities and OD quotas are consumed exactly after rounding/balancing.

In [30]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

import geopandas as gpd
import pandas as pd

from src import data_filepaths
from src.utils.loaders import load_boundary_pln_area, load_boundary_pln_region
from src.utils.population_distribution import (
    locate_demand_points, distribute_population, build_demand_data,
)
from src.utils.special_demand import (
    hotel_origins, merge_origin_points, distribute_special_demand,
)

ALLOW_EXPORT = True

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration
Dense pairs use the large size only if **both** endpoint populations exceed it and their remaining capacities and OD quota can accommodate it. All other groups use the small size. Large must be an integer multiple of small.

Counts are rounded to small-size multiples; the larger resident/job total is reduced proportionally to the smaller using largest remainders. Zero OD weights are soft targets, matching the legacy quota allocator. Same-point commutes are allowed. Driving metrics are straight-line estimates, not road routes.

In [ ]:
SMALL_SIZE = 50
LARGE_SIZE = 200
DISTANCE_SCALE = 10_000  # metres, EPSG:3414
RANDOM_SEED = 0
DRIVING_SPEED = 40 / 3.6  # metres per second

OUTPUT_PATH = Path(data_filepaths.OUT_DEMAND_DATA) / "demand_data_v2.json"

# Special demand configuration
INCLUDE_AIRPORT_DEMAND = True
INCLUDE_UNIVERSITY_DEMAND = True
SPECIAL_DEMAND_DISTANCE_SCALES = {
    "airport": 20_000,
    "university": 15_000,
}  # metres; independent of workforce DISTANCE_SCALE
SPECIAL_DEMAND_GROUP_ALLOCATION_MODES = {
    "airport": "large_first",  # default: large_first
    "university": "large_first",  # default: large_first
}  # use small_first to restore 50-person sampling before combining pairs
ORIGIN_MERGE_RADIUS = 200  # metres

In [32]:
demand_gdf = gpd.read_parquet(data_filepaths.AGGREGATED_DEMAND_POINTS)
df_ewarr = pd.read_parquet(data_filepaths.WORKPLACE_RESIDENCE_OD)
pln_area_gdf = load_boundary_pln_area()
pln_region_gdf = load_boundary_pln_region()
points_gdf = locate_demand_points(demand_gdf, pln_area_gdf, pln_region_gdf)

outside = points_gdf[['pln_area_id', 'pln_region_id']].isna().any(axis=1)
print(f'Excluded {outside.sum()} points outside planning boundaries:')
print(points_gdf.loc[outside, ['residents', 'workers']].sum())
points_gdf = points_gdf.loc[~outside].copy()
points_gdf[['residents', 'workers']].sum()

Excluded 1 points outside planning boundaries:
residents    199.929035
workers        0.000000
dtype: float64


residents    3.018854e+06
workers      3.013468e+06
dtype: float64

In [33]:
allocated_points, pops_df, od_people = distribute_population(
    points_gdf, df_ewarr.loc[:, pln_region_gdf.index],
    small_size=SMALL_SIZE, large_size=LARGE_SIZE,
    distance_scale=DISTANCE_SCALE, seed=RANDOM_SEED,
)
print(pd.DataFrame({
    'input': points_gdf[['residents', 'workers']].sum(),
    'allocated': allocated_points[['residents', 'workers']].sum(),
}))
print('Group sizes:', pops_df['size'].value_counts().to_dict())
od_people.head(8)

                  input  allocated
residents  3.018854e+06    3013150
workers    3.013468e+06    3013150
Group sizes: {200: 13162, 50: 7615}


,CR,ER,NER,NR,WR
AM,11300,7600,27850,16700,9900
BD,15300,40950,19300,8500,8400
BK,8400,3450,7950,7550,28500
BL,2950,1750,2100,2750,12450
BM,43450,20950,31400,14800,35350
BP,1500,600,1350,2050,11250
BS,11750,4350,13700,6050,6250
BT,5700,2750,4500,2500,9300


## Add non-workforce special demand
Run notebook 3's aggregate export and the enabled demand classes' SD4 exports first.
Universities use only existing residential origins, weighted by all-resident population.
Campus housing data is unavailable and is not estimated. Airports additionally use
hotels from `data_filepaths.HOTELS`, weighted by `TOTALROOMS` with stable `OBJECTID`s.

Each class uses its own scale in `SPECIAL_DEMAND_DISTANCE_SCALES`. Demand is
sampled in 200-person groups first by default, with 50-person groups only for
each destination/pool remainder (at most three). Configure each class in
`SPECIAL_DEMAND_GROUP_ALLOCATION_MODES`; `small_first` restores the former sampling.
Population and distance weighting remain the same, with more concentrated flows.
Demand budgets are independent of workforce capacities and OD quotas. Origins outside planning
boundaries are reported and excluded; invalid destination coordinates are errors.
Existing workforce groups and point coordinates remain unchanged.

The final JSON's `residents` and `jobs` count total modeled demand. Class, pool,
rounding, and distance-scale details are retained in companion exports.


In [ ]:
export_points = allocated_points
export_pops = pops_df

if INCLUDE_AIRPORT_DEMAND or INCLUDE_UNIVERSITY_DEMAND:
    residential_origins = gpd.read_parquet(
        data_filepaths.RESIDENTIAL_POPULATION_ORIGINS
    )
    origin_frames = [residential_origins]
    if INCLUDE_AIRPORT_DEMAND:
        origin_frames.append(hotel_origins(gpd.read_file(data_filepaths.HOTELS)))
    origins = gpd.GeoDataFrame(
        pd.concat(origin_frames),
        geometry="geometry",
        crs=3414,
    )
    located_origins = locate_demand_points(origins, pln_area_gdf, pln_region_gdf)
    outside_origins = (
        located_origins[["pln_area_id", "pln_region_id"]].isna().any(axis=1)
    )
    print("Excluded origin weights outside planning boundaries:")
    print(
        located_origins.loc[outside_origins]
        .groupby("pool_id")["population_weight"]
        .sum()
    )
    origin_points, origin_weights, origin_membership = merge_origin_points(
        allocated_points,
        located_origins.loc[~outside_origins],
        radius=ORIGIN_MERGE_RADIUS,
        seed=RANDOM_SEED,
    )
    located_origin_points = locate_demand_points(
        origin_points, pln_area_gdf, pln_region_gdf
    )
    if located_origin_points[["pln_area_id", "pln_region_id"]].isna().any().any():
        raise ValueError(
            "Merged origin points outside planning boundaries; inspect origin membership"
        )
    destination_frames = []
    if INCLUDE_AIRPORT_DEMAND:
        airport_demand = gpd.read_parquet(data_filepaths.AIRPORT_DEMAND)
        destination_frames.append(
            airport_demand.to_crs(3414).assign(category="airport")
        )
    if INCLUDE_UNIVERSITY_DEMAND:
        university_demand = gpd.read_parquet(data_filepaths.UNIVERSITY_DEMAND)
        destination_frames.append(
            university_demand.to_crs(3414).assign(category="university")
        )
    special_destinations = gpd.GeoDataFrame(
        pd.concat(destination_frames),
        geometry="geometry",
        crs=3414,
    )
    located_destinations = locate_demand_points(
        special_destinations, pln_area_gdf, pln_region_gdf
    )
    outside_destinations = (
        located_destinations[["pln_area_id", "pln_region_id"]].isna().any(axis=1)
    )
    if outside_destinations.any():
        raise ValueError(
            f"Special destinations outside planning boundaries: {located_destinations.index[outside_destinations].tolist()}"
        )
    export_points, special_pops, special_report = distribute_special_demand(
        origin_points,
        origin_weights,
        special_destinations,
        distance_scales=SPECIAL_DEMAND_DISTANCE_SCALES,
        group_allocation_modes=SPECIAL_DEMAND_GROUP_ALLOCATION_MODES,
        small_size=SMALL_SIZE,
        large_size=LARGE_SIZE,
        seed=RANDOM_SEED,
    )
    export_pops = pd.concat([pops_df, special_pops], ignore_index=True)
    print(special_report.to_string(index=False))
    print("Added demand by class:")
    print(special_pops.groupby("category")["size"].sum())
    print("Group counts by class and size:")
    print(special_pops.groupby(["category", "size"]).size())

Excluded origin weights outside planning boundaries:
Series([], Name: population_weight, dtype: float64)
         destination_id              pool_id    requested  allocated   category  distance_scale group_allocation_mode  rounding_delta
             AIR_SIN_T1 existing_residential  6721.461187       6700    airport           20000           large_first      -21.461187
             AIR_SIN_T1                hotel 10082.191781      10100    airport           20000           large_first       17.808219
             AIR_SIN_T2 existing_residential  7841.704718       7850    airport           20000           large_first        8.295282
             AIR_SIN_T2                hotel 11762.557078      11750    airport           20000           large_first      -12.557078
             AIR_SIN_T3 existing_residential  6161.339422       6150    airport           20000           large_first      -11.339422
             AIR_SIN_T3                hotel  9242.009132       9250    airport           2

In [35]:
demand_data = build_demand_data(export_points, export_pops, driving_speed=DRIVING_SPEED)
assert sum(p['residents'] for p in demand_data['points']) == sum(p['size'] for p in demand_data['pops'])
assert sum(p['jobs'] for p in demand_data['points']) == sum(p['size'] for p in demand_data['pops'])

In [36]:
if ALLOW_EXPORT:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with OUTPUT_PATH.open("w", encoding="utf-8") as output:
        json.dump(demand_data, output, allow_nan=False)
    print(
        f"Exported {len(demand_data['points'])} points and {len(demand_data['pops'])} groups to {OUTPUT_PATH}"
    )
    
    if INCLUDE_AIRPORT_DEMAND or INCLUDE_UNIVERSITY_DEMAND:
        origin_weights.to_parquet(OUTPUT_PATH.with_name("special_demand_origin_weights.parquet"), index=False)
        origin_membership.to_parquet(OUTPUT_PATH.with_name("special_demand_origin_membership.parquet"), index=False)
        for category, enabled in [("airport", INCLUDE_AIRPORT_DEMAND), ("university", INCLUDE_UNIVERSITY_DEMAND)]:
            if enabled:
                special_pops.loc[special_pops["category"].eq(category)].to_parquet(
                    OUTPUT_PATH.with_name(f"{category}_allocations.parquet"), index=False,
                )
                special_report.loc[special_report["category"].eq(category)].to_parquet(
                    OUTPUT_PATH.with_name(f"{category}_demand_report.parquet"), index=False,
                )


Exported 4286 points and 22230 groups to out\demand_data\demand_data_v2.json
